# Vetorização para cálculo de Document Frequency (DF)

In [ ]:
# Configurando o vetorizador CountVectorizer da biblioteca scikit-learn
vectorizer_df = CountVectorizer(
    ngram_range=(1, 1),
    min_df=5,
    binary=True
)

binary_matrix = vectorizer_df.fit_transform(df_corpus['clean_text'])

# Número de documentos em que cada termo aparece
doc_frequency = binary_matrix.sum(axis=0).A1
terms = vectorizer_df.get_feature_names_out()

print(f"Total de termos únicos: {len(terms)}")


# Tabela de ranking por DF

In [ ]:
total_docs = len(df_corpus)

table_df = pd.DataFrame({
    'term': terms,
    'doc_frequency': doc_frequency,
    'doc_frequency_pct': (doc_frequency / total_docs * 100).round(2)
}).sort_values('doc_frequency', ascending=False).reset_index(drop=True)

table_df.head(50)


# Parâmetros de candidatos a ruído

In [ ]:
EDGE_REVIEW_PCT = 15.0  # ajustável. Ponto de partida para triagem, não corte automático

targets_review = table_df[table_df['doc_frequency_pct'] >= EDGE_REVIEW_PCT].copy()

print(f"Termos acima de {EDGE_REVIEW_PCT}% de DF: {len(targets_review)}")
targets_review


# Curadoria

In [ ]:
targets_review['decision'] = ''  # preencher manualmente, ex: 'excluir', 'D'

NAME_FILE_SELECTION = 'insira-o-nome-do-arquivo-csv'
targets_review.to_csv(NAME_FILE_SELECTION, index=False, encoding='utf-8')

print(f"Arquivo de curadoria exportado: '{NAME_FILE_SELECTION}' com {len(targets_review)} termos para revisão.")


# Reconstrução do conjunto de stopwords pós-curadoria

In [ ]:
final_selection = pd.read_csv(f"{NAME_FILE_SELECTION}.csv")

CUSTOM_STOPWORDS = set(
    final_selection.loc[final_selection['decision'] == 'excluir', 'term']
)

print(f"CUSTOM_STOPWORDS construída com {len(CUSTOM_STOPWORDS)} termos.")
CUSTOM_STOPWORDS
